# 02 · Nhúng embedding bằng BGE-M3

**Quy trình:**

## Đầu ra

| File | Nội dung |
|---|---|
| `emb.npy` | Ma trận `float16` kích thước `[N, 1024]` |
| `texts.parquet` | Chỉ mục: `row_id` ↔ `question_id`, loại, cờ `has_answer`, nội dung |
| `manifest.json` | Tên mô hình, số chiều, ngày chạy — sáu tuần nữa nhìn lại sẽ cần |


In [1]:
!pip install -q FlagEmbedding 2>/dev/null || pip install -q sentence-transformers
print("xong")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.2 MB/s eta 0:00:00
xong


In [2]:
# ============ Cấu hình ============
from pathlib import Path
import json, gc, time
import numpy as np
import pandas as pd
import torch

DATA_DIR = Path("/kaggle/input/notebooks/phongngtun/ss-agent-getdata")
NB01_DIR = Path("/kaggle/input/notebooks/phongngtun/explore-longmemeval")   
OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "BAAI/bge-m3"
MAX_LEN    = 512          
BATCH      = 64           
SMOKE      = False        
N_SMOKE    = 20

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator!")

GPU: Tesla T4


In [3]:
# ============ Gom toàn bộ văn bản cần nhúng ============
with open(DATA_DIR / "longmemeval_s.json", encoding="utf-8") as f:
    data = json.load(f)

if SMOKE:
    data = data[:N_SMOKE]
    print(f"CHẾ ĐỘ THỬ — chỉ {len(data)} instance\n")

rows = []
for ex in data:
    qid = ex["question_id"]
    ans_ids = set(ex.get("answer_session_ids") or [])

    rows.append({"question_id": qid, "kind": "question", "session_id": "",
                 "session_idx": -1, "turn_idx": -1, "role": "",
                 "has_answer": False, "text": ex["question"]})

    for si, (sid, sess) in enumerate(zip(ex["haystack_session_ids"],
                                         ex["haystack_sessions"])):
        for ti, t in enumerate(sess if isinstance(sess, list) else []):
            if not isinstance(t, dict):
                continue
            txt = (t.get("content") or "").strip()
            if not txt:
                continue
            rows.append({
                "question_id": qid, "kind": "turn", "session_id": sid,
                "session_idx": si, "turn_idx": ti, "role": t.get("role", ""),
                # cờ vàng: chỉ tính là bằng chứng khi phiên đó nằm trong answer_session_ids
                "has_answer": bool(t.get("has_answer")) and sid in ans_ids,
                "text": txt,
            })

texts = pd.DataFrame(rows).reset_index(drop=True)
texts["row_id"] = texts.index
print(f"{len(texts):,} đoạn cần nhúng "
      f"({(texts.kind=='question').sum()} câu hỏi + {(texts.kind=='turn').sum():,} lượt)")
print(f"Số lượt là bằng chứng: {texts.has_answer.sum():,}")
print(f"Độ dài ký tự — trung vị {texts.text.str.len().median():.0f}, "
      f"p95 {texts.text.str.len().quantile(0.95):.0f}")

247,238 đoạn cần nhúng (500 câu hỏi + 246,738 lượt)
Số lượt là bằng chứng: 896
Độ dài ký tự — trung vị 426, p95 2886


In [4]:
# ============ Nạp mô hình ============
t0 = time.time()
try:
    from FlagEmbedding import BGEM3FlagModel
    _m = BGEM3FlagModel(MODEL_NAME, use_fp16=True)
    def embed(batch_texts):
        return _m.encode(batch_texts, batch_size=BATCH,
                         max_length=MAX_LEN)["dense_vecs"]
    backend = "FlagEmbedding"
except Exception as e:
    print("FlagEmbedding không dùng được, chuyển sang sentence-transformers:", e)
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(MODEL_NAME, device="cuda")
    _m.max_seq_length = MAX_LEN
    def embed(batch_texts):
        return _m.encode(batch_texts, batch_size=BATCH,
                         normalize_embeddings=True, show_progress_bar=False)
    backend = "sentence-transformers"

print(f"Đã nạp mô hình bằng {backend} — {time.time()-t0:.0f}s")

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Đã nạp mô hình bằng FlagEmbedding — 57s


In [5]:
# ============ Nhúng theo lô, lưu từng chunk ============
CHUNK = 4096                                   # lưu tạm sau mỗi chunk
all_texts = texts["text"].tolist()
n = len(all_texts)
parts, t0 = [], time.time()

for start in range(0, n, CHUNK):
    part = embed(all_texts[start:start + CHUNK])
    parts.append(np.asarray(part, dtype=np.float16))
    done = min(start + CHUNK, n)
    el = time.time() - t0
    print(f"  {done:>7,}/{n:,}  ({done/n:5.1%})  "
          f"{el:6.0f}s trôi qua  ·  còn ~{el/done*(n-done):5.0f}s", flush=True)
    gc.collect(); torch.cuda.empty_cache()

emb = np.vstack(parts)
del parts; gc.collect()

# chuẩn hóa L2 để cosine = tích vô hướng (nhanh hơn nhiều ở notebook 03)
norms = np.linalg.norm(emb.astype(np.float32), axis=1, keepdims=True)
emb = (emb.astype(np.float32) / np.clip(norms, 1e-9, None)).astype(np.float16)

print(f"\nXong: {emb.shape}  ·  {emb.nbytes/1e6:.0f} MB  ·  {time.time()-t0:.0f}s")

Chunks: 100%|██████████| 2/2 [00:18<00:00,  9.35s/it]

    4,096/247,238  ( 1.7%)      39s trôi qua  ·  còn ~ 2316s



Chunks: 100%|██████████| 2/2 [00:18<00:00,  9.06s/it]

    8,192/247,238  ( 3.3%)      57s trôi qua  ·  còn ~ 1676s



Chunks: 100%|██████████| 2/2 [00:19<00:00,  9.69s/it]

...


Chunks: 100%|██████████| 2/2 [00:21<00:00, 10.58s/it]

  245,760/247,238  (99.4%)    1328s trôi qua  ·  còn ~    8s



Chunks: 100%|██████████| 2/2 [00:08<00:00,  4.27s/it]

  247,238/247,238  (100.0%)    1337s trôi qua  ·  còn ~    0s



Xong: (247238, 1024)  ·  506 MB  ·  1341s


In [6]:
# ============ Lưu ============
np.save(OUT_DIR / "emb.npy", emb)
texts.drop(columns=[]).to_parquet(OUT_DIR / "texts.parquet", index=False)

manifest = {
    "model": MODEL_NAME,
    "backend": backend,
    "dim": int(emb.shape[1]),
    "n_rows": int(emb.shape[0]),
    "dtype": "float16",
    "l2_normalized": True,
    "max_length": MAX_LEN,
    "smoke": SMOKE,
    "n_instances": len(data),
    "created_at": pd.Timestamp.utcnow().isoformat(),
}
with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("\nCÁC FILE ĐÃ LƯU:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:22s} {p.stat().st_size/1e6:8.1f} MB")

{
  "model": "BAAI/bge-m3",
  "backend": "FlagEmbedding",
  "dim": 1024,
  "n_rows": 247238,
  "dtype": "float16",
  "l2_normalized": true,
  "max_length": 512,
  "smoke": false,
  "n_instances": 500,
  "created_at": "2026-09-21T10:15:34.178097+00:00"
}

CÁC FILE ĐÃ LƯU:
  __notebook__.ipynb          0.8 MB
  emb.npy                   506.3 MB
  manifest.json               0.0 MB
  texts.parquet             122.5 MB


In [7]:
# ============ Kiểm tra nhanh: embedding có hợp lý không ============
# Với vài câu hỏi, cosine tới câu bằng chứng phải CAO HƠN cosine tới lượt ngẫu nhiên.
rng = np.random.default_rng(0)
qs = texts[texts.kind == "question"].head(5)

for _, q in qs.iterrows():
    qv = emb[q.row_id].astype(np.float32)
    same = texts[(texts.question_id == q.question_id) & (texts.kind == "turn")]
    pos = same[same.has_answer]
    neg = same[~same.has_answer]
    if not len(pos) or not len(neg):
        continue
    s_pos = float(np.mean(emb[pos.row_id.values].astype(np.float32) @ qv))
    s_neg = float(np.mean(emb[rng.choice(neg.row_id.values,
                                         min(50, len(neg)), replace=False)]
                          .astype(np.float32) @ qv))
    ok = "OK " if s_pos > s_neg else "NGỜ"
    print(f"  [{ok}] bằng chứng {s_pos:.3f}  vs  ngẫu nhiên {s_neg:.3f}   "
          f"{q.text[:60]}")

print("\nNếu dòng nào 'NGỜ' cũng xuất hiện thường xuyên thì kiểm tra lại "
      "cách gắn cờ has_answer trước khi chạy toàn bộ.")

  [OK ] bằng chứng 0.508  vs  ngẫu nhiên 0.339   What degree did I graduate with?
  [OK ] bằng chứng 0.716  vs  ngẫu nhiên 0.332   How long is my daily commute to work?
  [OK ] bằng chứng 0.805  vs  ngẫu nhiên 0.342   Where did I redeem a $5 coupon on coffee creamer?
  [OK ] bằng chứng 0.656  vs  ngẫu nhiên 0.307   What play did I attend at the local community theater?
  [OK ] bằng chứng 0.622  vs  ngẫu nhiên 0.337   What is the name of the playlist I created on Spotify?

Nếu dòng nào 'NGỜ' cũng xuất hiện thường xuyên thì kiểm tra lại cách gắn cờ has_answer trước khi chạy toàn bộ.
